# Cluster analysis and validation

This notebook explores the semantic structure of Last.fm genre tags using TF-IDF, UMAP and K-Means clustering. Initial clusters are iteratively reviewed and consolidated into four interpretable macro-clusters used in the final longitudinal analysis.


## 1. Load libraries

In [ ]:
# env: music-umap as umap needed numpy < 2
import pandas as pd
import umap
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import ast

## 2. Load clustered dataset

In [ ]:
model_df = pd.read_csv('../data/processed/lastfm_music_clusters_final.csv')

## 3. Dataset overview

In [ ]:
# Analyzed df contains ranking of unique tracks by play count, with associated tags and cluster assignments
# It doesn't contain the whole list of scrobbles or the full tag history, but it has enough data to visualize the clusters and understand their characteristics.
# All scrobbles will be in raw data
model_df.head()

In [ ]:
model_df.shape

In [ ]:
top_50_all_time = model_df["artist_clean"].value_counts().head(50)

In [ ]:
top_50_all_time

In [ ]:
model_df.columns.tolist()

In [ ]:
model_df["artist_clean"].value_counts().head(20)

In [ ]:
model_df.groupby("artist_clean")["track_clean"].nunique().sort_values(ascending=False).head(20)

## 4. TF-IDF → UMAP projection

In [ ]:
# TFIDFVectorizer breaks tags with spaces, so we need to join them with underscores

model_df["tags_filtered"] = model_df["tags_filtered"].apply(ast.literal_eval)

In [ ]:
def join_tags(tags):
    return " ".join([t.replace(" ", "_") for t in tags])

model_df["tags_str"] = model_df["tags_filtered"].apply(join_tags)

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(model_df["tags_str"])

In [ ]:
reducer = umap.UMAP(
    n_neighbors=30,      # local structure
    min_dist=0.05,        # greater clustering
    metric="cosine",#  KEY for TF-IDF
    densmap=True,
    random_state=42
)
X_umap = reducer.fit_transform(X)

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=model_df["cluster"],
    cmap="tab10",
    alpha=0.6
)

plt.legend(*scatter.legend_elements(), title="Clusters")
plt.title("UMAP Music Map")
plt.show()

## 5. Cluster inspection

In [ ]:
mask = model_df["cluster"] == 0

plt.scatter(X_umap[:,0], X_umap[:,1], color="lightgray", alpha=0.2)
plt.scatter(X_umap[mask,0], X_umap[mask,1], color="red", alpha=0.7)
plt.show()

## 6. Initial cluster interpretation

The initial K-Means clusters were inspected using representative artists and tag distributions to assess semantic coherence.

In [ ]:
# Evolution of music taste over time

df = (
    model_df.groupby(["year_file", "cluster_name"])
    .size()
    .unstack(fill_value=0)
)

df_pct = df.div(df.sum(axis=1), axis=0)

df_pct.plot(kind="area", figsize=(12, 6), colormap="tab10")
plt.title("Music Taste Evolution Over Time")
plt.ylabel("Share")
plt.xlabel("Year")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
cluster_names_v2 = {
    0: "Americana / indie folk",
    1: "British alternative rock",
    2: "Ambient / trip-hop / downtempo",
    3: "Alternative rock mainstream",
    4: "Neoclassical / soundtrack",
    5: "Art indie / dream pop",
    6: "Soul / R&B / funk",
    7: "Post-punk / goth / new wave"
}

model_df["cluster_name_v2"] = model_df["cluster"].map(cluster_names_v2)

In [ ]:
for cluster in model_df["cluster_name_v2"].unique():
    print("\n")
    print(cluster)

    print(
        model_df[
            model_df["cluster_name_v2"] == cluster
        ]["artist_clean"]
        .value_counts()
        .head(10)
    )

In [ ]:
cluster_names_v3 = {
    0: "Alternative & Britrock",
    1: "Rock Canon & Alternative Classics",
    2: "Grunge & 90s Alternative",
    3: "Dark Alternative & Post-Punk",
    4: "Ambient & Trip-Hop Electronic",
    5: "Art Rock & Indie Experimental",
    6: "Americana & Singer-Songwriter",
    7: "Dream Pop & Modern Indie"
}

model_df["cluster_name_v3"] = model_df["cluster"].map(cluster_names_v3)

In [ ]:
for cluster in model_df["cluster_name_v3"].unique():
    print("\n")
    print(cluster)

    print(
        model_df[
            model_df["cluster_name_v3"] == cluster
        ]["artist_clean"]
        .value_counts()
        .head(10)
    )

In [ ]:
model_df["cluster_name_v3"].value_counts()

In [ ]:
model_df.groupby("cluster_name_v3")["artist_clean"].nunique()

In [ ]:
model_df.groupby("cluster_name_v3").size()

In [ ]:
(
    model_df["cluster_name_v3"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

In [ ]:
model_df.groupby("cluster_name_v3")["tags_filtered"].count()

## 7. Macro-cluster consolidation

Because several initial clusters overlapped stylistically, they were merged into four broader categories used in the longitudinal analysis.

In [ ]:
macro_map = {
    "Dark Alternative & Post-Punk": "Alternative Core",

    "Rock Canon & Alternative Classics": "Rock & Britrock",
    "Alternative & Britrock": "Rock & Britrock",
    "Grunge & 90s Alternative": "Rock & Britrock",

    "Ambient & Trip-Hop Electronic": "Electronic & Ambient",
    "Art Rock & Indie Experimental": "Electronic & Ambient",
    "Dream Pop & Modern Indie": "Electronic & Ambient",

    "Americana & Singer-Songwriter": "Singer-Songwriter"
}

model_df["macro_cluster_v2"] = model_df["cluster_name_v3"].map(macro_map)

In [ ]:
model_df["macro_cluster_v2"].value_counts(normalize=True)

In [ ]:
# Years 2007 and 2026 are incomplete, so we will exclude them from the analysis
analysis_df = model_df[
    (model_df["year_file"] >= 2008)
    & (model_df["year_file"] <= 2025)
]

## 8. Final macro-clusters

The initial K-Means solution produced several overlapping musical categories. After manual inspection, the clusters were consolidated into four broader macro-clusters that better represent long-term listening patterns and are used throughout the final longitudinal analysis. The following visualization illustrates how the relative share of these macro-clusters changed over time.

In [ ]:
macro_year = (
    model_df
    .groupby(["year_file", "macro_cluster_v2"])
    .size()
    .unstack(fill_value=0)
)

macro_pct = (
    pd.crosstab(
        analysis_df["year_file"],
        analysis_df["macro_cluster_v2"],
        normalize="index"
    )
)

macro_pct.head()

In [ ]:
macro_pct.tail()

In [ ]:
CLUSTER_COLORS = {
    "Alternative Core": "#4E79A7",      
    "Electronic & Ambient": "#F28E2B",            
    "Rock & Britrock": "#59A14F",                  
    "Singer-Songwriter": "#E15759"             
}

In [ ]:
colors = [CLUSTER_COLORS[c] for c in macro_pct.columns]

plt.figure(figsize=(12,6))

macro_pct.plot.area(
    figsize=(12,6),
    alpha=0.8,
    color=colors
)

plt.title("Evolution of Musical Preferences")
plt.ylabel("Share of Listening")
plt.xlabel("Year")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## 9. Export processed dataset

In [ ]:
# Save analysis dataset

analysis_df = analysis_df.rename(
    columns={
        "macro_cluster_v2": "macro_cluster"
    }
)

analysis_df = analysis_df.drop(
    columns=[
        "cluster_name_v2",
        "cluster_name_v3"
    ]
)

analysis_df.to_csv(
    "../data/processed/final/music_taste_analysis_ready.csv",
    index=False
)

import pickle

with open(
    "../data/processed/final/music_taste_analysis_ready.pkl",
    "wb"
) as f:
    pickle.dump(analysis_df, f)

In [ ]:
# how to load pickle, if needed:
with open(
    "../data/processed/final/music_taste_analysis_ready.pkl",
    "rb"
) as f:
    model_df = pickle.load(f)

In [ ]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

## Output

Output: `data/processed/final/music_taste_analysis_ready.csv`

Used in **07_music_taste_evolution.ipynb**